# 07 · Defensive analysis

Closeouts, coverage and matchups — and what the data does not support.

In [1]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config as C
pd.set_option("display.width", 160, "display.max_columns", 60)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True,
                     "grid.alpha": .25, "figure.dpi": 110})
E = C.PROCESSED_DIR / "events"; D = C.DERIVED_DIR
rd  = lambda n: pd.read_parquet(D / f"{n}.parquet")
rev = lambda n: pd.read_parquet(E / f"{n}.parquet")
print("project:", C.PROJECT_ROOT.name)

project: basketball-intelligence


In [2]:
co = rd("closeouts_features")
print(f"{len(co)} closeouts | start {co.start_distance.mean():.1f} ft -> end "
      f"{co.end_distance.mean():.1f} ft | rate {co.close_rate_fps.mean():.1f} ft/s")
co.groupby("bhr_action").agg(n=("id","size"), start=("start_distance","mean"),
  end=("end_distance","mean"), rate=("close_rate_fps","mean"),
  pts=("total_points_allowed","mean"), stop=("stop","mean"), foul=("fouled","mean")).round(3)

601 closeouts | start 14.3 ft -> end 5.4 ft | rate 7.0 ft/s


,n,start,end,rate,pts,stop,foul
bhr_action,,,,,,,
drive,127,13.233,3.936,7.397,0.606,0.591,0.165
other,74,14.110,5.002,6.795,0.608,0.541,0.230
pass,117,13.728,5.184,6.484,0.359,0.863,0.000
shot,282,15.060,6.228,7.168,0.908,0.677,0.014


In [3]:
from scipy import stats
tight, loose = co[co.arrived_tight], co[~co.arrived_tight]
print(f"tight (<=4 ft, n={len(tight)}): {tight.total_points_allowed.mean():.3f} pts, "
      f"{tight.fouled.mean():.1%} fouls, {tight.stop.mean():.1%} stops")
print(f"loose     (n={len(loose)}): {loose.total_points_allowed.mean():.3f} pts, "
      f"{loose.fouled.mean():.1%} fouls, {loose.stop.mean():.1%} stops")
chi2, p, *_ = stats.chi2_contingency(pd.crosstab(co.arrived_tight, co.fouled))
print(f"foul-rate difference: chi2 p = {p:.4f}")
b = pd.cut(co.end_distance, [0,2,3,4,5,6,8,30])
co.groupby(b, observed=True).agg(n=("id","size"), pts=("total_points_allowed","mean"),
  stop=("stop","mean"), foul=("fouled","mean")).round(3)

tight (<=4 ft, n=199): 0.558 pts, 11.6% fouls, 67.8% stops
loose     (n=402): 0.769 pts, 4.7% fouls, 67.9% stops
foul-rate difference: chi2 p = 0.0035


,n,pts,stop,foul
end_distance,,,,
"(0, 2]",31,0.645,0.581,0.194
"(2, 3]",63,0.603,0.603,0.143
"(3, 4]",105,0.505,0.752,0.076
"(4, 5]",105,0.705,0.667,0.076
"(5, 6]",108,0.639,0.722,0.046
"(6, 8]",105,0.829,0.667,0.038
"(8, 30]",81,0.975,0.642,0.025


## Coverage mix by defending team

In [4]:
pk = rd("picks_features")
teams = dict(zip(*pd.read_parquet(C.PROCESSED_DIR/"dim/teams.parquet")[["team_id","team_name"]].values.T))
pk["def_team"] = pk.def_team_id.map(teams)
mix = pk.groupby("def_team").agg(picks=("id","size"), ppp_allowed=("pick_points","mean"),
  over=("bhr_def_type", lambda s: s.eq("over").mean()),
  under=("bhr_def_type", lambda s: s.eq("under").mean()),
  switch=("bhr_def_type", lambda s: s.eq("switch").mean())).sort_values("ppp_allowed")
mix.round(3)

,picks,ppp_allowed,over,under,switch
def_team,,,,,
Coviran Granada,59,0.271,0.712,0.119,0.153
Leche Rio Breogan,134,0.291,0.739,0.134,0.112
Bitci Baskonia,78,0.346,0.679,0.090,0.179
BAXI Manresa,76,0.355,0.750,0.145,0.105
Club Joventut Badalona,92,0.359,0.728,0.141,0.130
San Pablo Burgos,87,0.368,0.552,0.218,0.230
BC Barcelona,69,0.406,0.638,0.130,0.217
CB UCAM Murcia,83,0.410,0.518,0.120,0.313
Dreamland Gran Canaria,139,0.417,0.504,0.367,0.115


## Matchup workload

Read as activity, not as a rating.

In [5]:
mw = rd("matchup_workload")
mw.nlargest(15, "minutes_guarding")[["player_name","team_name","games","assignments",
  "minutes_guarding","distinct_opponents","shots_allowed","points_allowed",
  "points_allowed_per_36"]].round(2)

,player_name,team_name,games,assignments,minutes_guarding,distinct_opponents,shots_allowed,points_allowed,points_allowed_per_36
69,Devin Robinson,Casademont Zaragoza,2,206,18.87,19,20,19,36.25
176,Santiago Yusta,Casademont Zaragoza,2,213,17.26,19,16,21,43.79
52,Trae Bell-Haynes,Casademont Zaragoza,2,195,16.28,19,21,17,37.59
104,Aleksandar Aranitovic,Leche Rio Breogan,2,181,15.97,20,17,18,40.57
43,Francisco Alonso,Leche Rio Breogan,2,163,15.26,18,8,11,25.94
3,Isaiah Wong,Dreamland Gran Canaria,2,156,14.93,19,16,14,33.75
72,Nico Brussino,Dreamland Gran Canaria,2,147,14.88,20,10,9,21.77
83,DeWayne Russell,Leche Rio Breogan,2,138,14.57,18,15,12,29.65
54,Kassius Robertson,Dreamland Gran Canaria,2,147,14.42,19,21,22,54.94
84,Mihajlo Andric,Leche Rio Breogan,2,161,13.85,18,13,14,36.39
